# TabPFN-3.5 at 75,000 training rows

Uses every signed training row. `auto` selects cached fitting on an L4 or larger GPU and low-memory fitting on a T4. An A100 is not required by this trial contract, but exact success and runtime still depend on the assigned Colab GPU and host RAM.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_REPO_URL='https://github.com/selimsidan/tfm-credit-benchmark.git'
GITHUB_BRANCH='main'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/tfm_credit_benchmark'
SOURCE_EXPERIMENT_NAME='lending_club_900k'
TRIAL_EXPERIMENT_NAME='lending_club_75k_trial'
FORCE_RERUN=False
RUN_TESTS=True
TABPFN_FIT_MODE='auto'  # auto, fit_with_cache, fit_preprocessors, or low_memory
PREDICT_VALIDATION=False
# ==================================================================


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, subprocess, sys
drive.mount('/content/drive'); repo=Path('/content/tfm-credit-benchmark')
if (repo/'.git').is_dir(): subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',GITHUB_BRANCH],check=True)
else: subprocess.run(['git','clone','--branch',GITHUB_BRANCH,'--single-branch',GITHUB_REPO_URL,str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
src_path=str(repo/'src')
if src_path not in sys.path: sys.path.insert(0,src_path)
import tfmbench
print('tfmbench package:',tfmbench.__file__)
try: tabpfn_token=userdata.get('TABPFN_TOKEN')
except Exception: tabpfn_token=None
if not tabpfn_token: raise RuntimeError('Missing TABPFN_TOKEN. Accept the TabPFN-3.5 license and grant this notebook access to the Colab secret.')
os.environ['TABPFN_TOKEN']=tabpfn_token; tabpfn_token=None
print('Repository commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())


In [ ]:
import json
from tfmbench.resources import require_cuda
from tfmbench.trial import TrialConfig, verify_trial_manifest
if RUN_TESTS: subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
hardware=require_cuda('TabPFN-3.5 75k trial'); print(json.dumps(hardware,indent=2))
gpu_name=str(hardware.get('gpu_name') or '')
vram_gib=float(hardware.get('gpu_vram_bytes') or 0)/1024**3
if TABPFN_FIT_MODE=='auto': TABPFN_FIT_MODE='fit_with_cache' if vram_gib>=20 else 'low_memory'
print('Resolved TabPFN fit mode:',TABPFN_FIT_MODE)
if 'T4' in gpu_name: print('T4 selected: low_memory is safest but can be substantially slower. Use a High-RAM runtime if available.')
elif 'L4' in gpu_name: print('L4 selected: recommended starting point for this 75k trial.')
else: print('GPU was not explicitly profiled; the run will record memory and fail without changing the training context if capacity is insufficient.')
config=TrialConfig(DRIVE_OUTPUT_DIR,TRIAL_EXPERIMENT_NAME,SOURCE_EXPERIMENT_NAME)
manifest=verify_trial_manifest(config); print('Verified trial manifest:',manifest['manifest_hash'])


In [ ]:
from tfmbench.trial import run_trial_method
table=run_trial_method('tabpfn35',DRIVE_OUTPUT_DIR,SOURCE_EXPERIMENT_NAME,TRIAL_EXPERIMENT_NAME,force_rerun=FORCE_RERUN,tabpfn_fit_mode=TABPFN_FIT_MODE,predict_validation=PREDICT_VALIDATION)


In [ ]:
display(table)
print('TabPFN results:',config.experiment_root/'tabpfn35'/'n75000')
print('Next notebook: 10_compare_75k.ipynb')
